In [ ]:
import functools
import numpy as np
import qiskit as qk
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
from scipy.integrate import simpson

import sys
sys.path.append("../")
import pulse_simulator as ps

import matplotlib.pyplot as plt

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()

dt = backend.configuration().dt * ns
duration = 225 # dt

# Can be acquired with backend.configuration().timing_constraints['granularity']
granularity = 16
GATE_TIME_SI_UNITS = duration * dt / ns

def get_closest_multiple_of(value, base_number):
    return int(value + base_number/2) - (int(value + base_number/2) % base_number)

def get_closest_multiple_of_16(num, granularity=16):
    return get_closest_multiple_of(num, granularity)

registers = [0, 1]

# Variables
# NOTE: If the Rabi rates are True, you have to calibrate!
config_vars = ps.backend_simulation_vars(backend, rabi=True, units=Units)

# Carrier frequencies of each control line
carriers = ps.backend_carriers(backend, config_vars)

config_vars;

In [ ]:
pico_a = ps.from_csv("../pico-pulses/saved-pulses-2024-01-24/a_single_qubit_gateset_R1e-3.csv")
fig, ax = plt.subplots(figsize=[5,3])
for row in pico_a:
    ax.plot(row)

In [ ]:
np.trapz(pico_a[0], dx=dt) / np.pi

In [ ]:
# signal_samples = ps.get_samples(ps.gaussian_envelope(duration, dt=dt, angle=np.pi / 2))

signal_samples = ps.resample(
    duration,
    ps.get_samples(
        ps.custom_envelope(len(pico_a[0]), pico_a[0], dt=dt, angle=(1.5) * np.pi)
    )
)

# Rescale so new steps are size 1
gate_lookup = {
    "custom": ps.scalable_symbolic_pulse(duration, signal_samples, "custom")
}

In [ ]:
np.trapz(ps.get_samples(gate_lookup["custom"]).real, dx=dt) / np.pi

In [ ]:
samples = ps.get_samples(gate_lookup["custom"])

fig, ax = plt.subplots(figsize=(5, 3))
# Rescale to match the original length
scale = pico_a.shape[1] / duration
plt.plot(np.arange(0, duration) * scale, samples.real / scale, label="custom")
plt.plot(np.arange(pico_a.shape[1]), pico_a[0], label="pico", ls=':')
plt.legend()

In [ ]:
gates = {0: "custom"}

before = simpson(ps.get_samples(gate_lookup["custom"]).real, dx=dt)
print(f"Before {before / np.pi} π")

with qk.pulse.build(name="Current moment", backend=backend) as rabi_sched:
    for i, gate in gates.items():
        channel = qk.pulse.DriveChannel(i)

        samples = get_closest_multiple_of_16(
            qk.pulse.seconds_to_samples(GATE_TIME_SI_UNITS)
        )

        # TODO: Can we change the samples in the waveform returned by SymbolicPulse?

        qk.pulse.play(gate_lookup[gate], channel)

after = np.trapz(ps.get_samples(rabi_sched.instructions[0][1].pulse).real, dx=dt)
print(f"After {after / np.pi} π")

Two options: Circuit or pulse schedule.

In [ ]:
# Excecute rabi schedules
drive_amp = qk.circuit.Parameter('drive_amp')
drive_amps = np.linspace(0.8, 1.2, 51)
exp_rabi_scheds = [rabi_sched.assign_parameters({drive_amp: a}, inplace=False) for a in drive_amps]

In [ ]:
# Add rabi gate to circuit
rabi_gate = qk.circuit.Gate("rabi", 1, [drive_amp])
qc_rabi = qk.QuantumCircuit(1, 1)
qc_rabi.append(rabi_gate, [0])
qc_rabi.measure(0, 0)

# Register a low-level, custom pulse definition for the given gate.
qc_rabi.add_calibration(rabi_gate, (0,), rabi_sched, [drive_amp])

# Execute rabi circuits
drive_amps = np.linspace(.8, 1.2, 51)
exp_rabi_circs = [qc_rabi.assign_parameters({drive_amp: a}, inplace=False) for a in drive_amps]

In [ ]:
rabi_schedule = qk.schedule(exp_rabi_circs[-1], backend)
rabi_schedule.draw(backend=backend)

To simulate, use fake 2 qubit?

In [ ]:
fake_2q = qk_fp.FakeOpenPulse2Q()

In [ ]:
num_shots_per_point = 1024

job = fake_2q.run(
    exp_rabi_circs, meas_level=1, meas_return="avg", shots=num_shots_per_point
)

# job.submit()

In [ ]:
# Fails because fake backends don't have pulse gate support
job.result()

# Miscellany

In [ ]:
# Check that resample works.
plt.step(np.linspace(0, duration, samples, endpoint=False), ps.resample(samples, gate_lookup[gate].samples).real)
plt.step(np.arange(0, duration), gate_lookup[gate].samples.real)